**EDA conclusions**

**- Class distribution:** 100% balanced across negative, positive, and mixed classes, assuming each class contains approximately one-third of the dataset.

**- Missing values:** None.

**- Duplicates:** Zero.

**- Outliers:** Few or none in review length.

**- Text length:** Reviews have varying character lengths, providing some diversity in text input.

In [ ]:

import pandas as pd



In [ ]:

# Update this path to your actual CSV or Excel location
file_path = 'ar_reviews_100k.tsv'

# Using read_csv with tab separator because the file is a TSV
df = pd.read_csv(file_path, sep='\t')

print("Dataset loaded successfully!")
display(df.head())


In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]:,}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"Total cells: {df.size:,}")

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataFrame information:")
df.info()

In [ ]:
print("Missing values:")
display(df.isnull().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())

print("Duplicate rows percentage:",
      f"{df.duplicated().mean() * 100:.2f}%")

In [ ]:
# Inspect the label counts for each column to identify the sentiment column
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))

In [ ]:
import matplotlib.pyplot as plt

LABEL_COLUMN = "label"  # Replace with the actual column name

class_counts = df[LABEL_COLUMN].value_counts(dropna=False)

display(pd.DataFrame({
    "count": class_counts,
    "percentage": (class_counts / len(df) * 100).round(2)
}))

class_counts.plot(kind="bar", figsize=(8, 4))
plt.title("Sentiment Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Reviews")
plt.show()


In [ ]:
TEXT_COLUMN = "text"  # Replace with the actual review column name

df["char_length"] = df[TEXT_COLUMN].astype("string").str.len()
df["word_count"] = df[TEXT_COLUMN].astype("string").str.split().str.len()

display(df[["char_length", "word_count"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
))

df["word_count"].plot(kind="box", vert=False, figsize=(10, 3))
plt.title("Review Length Outliers")
plt.xlabel("Word Count")
plt.show()

display(df.nlargest(10, "word_count")[[TEXT_COLUMN, LABEL_COLUMN, "word_count"]])
display(df.nsmallest(10, "word_count")[[TEXT_COLUMN, LABEL_COLUMN, "word_count"]])

In [ ]:
normalized_text = (
    df[TEXT_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
)

temp = pd.DataFrame({
    "review": normalized_text,
    "label": df[LABEL_COLUMN]
})

label_counts = temp.groupby("review")["label"].nunique()
conflicting_reviews = label_counts[label_counts > 1]

print("Unique review texts:", normalized_text.nunique())
print("Duplicate review occurrences:", normalized_text.duplicated().sum())
print("Reviews with conflicting labels:", len(conflicting_reviews))

In [ ]:
# Install required library in the notebook environment
!pip install -q arabert transformers

In [ ]:
from arabert.preprocess import ArabertPreprocessor
from transformers import AutoTokenizer

MODEL = "aubmindlab/bert-base-arabertv02"
prep = ArabertPreprocessor(model_name=MODEL)
tok = AutoTokenizer.from_pretrained(MODEL)

sample = df.sample(5000, random_state=42)
lengths = pd.Series([len(tok(prep.preprocess(t))["input_ids"]) for t in sample["text"]])

print(lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
for n in (64, 128, 192, 256, 512):
    print(f"max_length {n}: {(lengths <= n).mean():.1%} of reviews fit without truncation")

In [ ]:
sample["tokens"] = lengths.values
print(sample.groupby("label")["tokens"].median())

In [ ]:
for lab in ["Negative", "Mixed", "Positive"]:
    print("=====", lab)
    for t in df[df["label"] == lab]["text"].sample(5, random_state=1):
        print("-", t[:200])

In [ ]:
short = df[df["word_count"] <= 2]
print(len(short))
print(short["label"].value_counts())

In [ ]:
df["hotel_words"] = df["text"].str.contains("فندق|الفندق|غرفة|الغرفة|النظافة")
df["book_words"] = df["text"].str.contains("كتاب|الكتاب|رواية|الرواية|المؤلف|الكاتب")
print(df.groupby("label")[["hotel_words", "book_words"]].mean().round(3))

# EDA conclusions: ar_reviews_100k

## Data
- **Size:** 99,999 reviews, 2 columns (`label`, `text`).
- **Quality:** no missing values, no exact duplicates, and no review text appears with two different labels.
- **Classes:** Positive, Mixed and Negative have exactly 33,333 reviews each. This balance is artificial. Real traffic will not be balanced, so test scores on this data will look better than live performance, and we must monitor the live class mix.
- **Content:** the reviews cover several topics, including hotels and books. A keyword check found a similar topic mix in all three classes (hotel words in 21-23% and book words in 33-37% of each class), so topic does not obviously give away the label. This check is rough and does not show what the remaining reviews are about.
- **Label meaning:** the examples suggest "Mixed" means a middling or two-sided opinion, probably from a middle star rating. This is a hypothesis, not a verified fact. "Mixed" is not the same as "neutral", so the README must say how we name this class.

## Review length
- **Characters:** median 144, 95th percentile 1,059, maximum 8,163. The distribution has a long right tail, so the mean (298) is much larger than the median.
- **AraBERT tokens** (5,000-review sample): median 38, 90th percentile 165, 99th percentile 565, maximum 1,600. The model can read at most 512 tokens.
- **Share of reviews that fit without truncation:** 64 tokens: 68.6%, 128 tokens: 85.2%, 192 tokens: 92.1%, 256 tokens: 95.0%, 512 tokens: 98.7%.
- **By class** (median tokens): Mixed 40, Negative 40, Positive 34. The gap is small, so length alone is a weak clue.

## Short reviews
- 731 reviews (0.73%) have one or two words. Negative is the most common class among them (329), then Positive (228) and Mixed (174).
- We keep them, because the API will receive short texts too. The model will probably be weaker on them, and we will measure that separately.

## Decisions
1. Start with `max_length = 128`, and compare it with `max_length = 256`.
2. Use the same AraBERT preprocessing in training and in the API.
3. Report per-class results as well as overall accuracy, using macro-F1 as the main metric.
4. In the README, state the artificial balance, the mixed topics and the meaning of "Mixed" as known limitations.

## Limits of this analysis
- Token statistics come from a random sample of 5,000 reviews, so each percentage is accurate to about a point.
- Duplicate checks cover exact matches and matches after trimming spaces and lowercasing. They do not catch near-duplicates or Arabic spelling variants.
